In [2]:
import pandas as pd
from pathlib import Path

# path = Path("../data/processed/pam_parana_features.parquet")
path = Path("../data/processed/pam_parana_consolidado.parquet")

df = pd.read_parquet(path)
df.head()

# df_raw.groupby(["produto","municipio_codigo"])[["valor_producao", "area_colhida"]].mean()

,municipio_codigo,municipio_nome,ano,rendimento_medio,quantidade_produzida,valor_producao,area_colhida,area_plantada,produto
0,4100103,Abatiá - PR,2010,2590,17820,8713,6880,6880,soja
1,4100103,Abatiá - PR,2011,3350,23048,15557,6880,6880,soja
2,4100103,Abatiá - PR,2012,2320,16472,14001,7100,7100,soja
3,4100103,Abatiá - PR,2013,3100,23250,20576,7500,7500,soja
4,4100103,Abatiá - PR,2014,1460,12330,12774,8445,8445,soja


In [3]:
# Market Share: % de contribuição do município na produção total do grão no estado naquele ano.
import numpy as np
df["total_prod_ano"] = df.groupby(["ano", "produto"])["quantidade_produzida"].transform("sum") # produção total do grão X de cada ano
df["market_share"] = df["quantidade_produzida"]/ df["total_prod_ano"].replace(0, np.nan) # evita a divisão por 0, resultando em Nan
df["market_share"] = df["market_share"].fillna(0.0) # Evita quebrar o modelo caso houver Nan

In [4]:
df.head()

,municipio_codigo,municipio_nome,ano,rendimento_medio,quantidade_produzida,valor_producao,area_colhida,area_plantada,produto,total_prod_ano,market_share
0,4100103,Abatiá - PR,2010,2590,17820,8713,6880,6880,soja,14091829,0.001265
1,4100103,Abatiá - PR,2011,3350,23048,15557,6880,6880,soja,15457911,0.001491
2,4100103,Abatiá - PR,2012,2320,16472,14001,7100,7100,soja,10937896,0.001506
3,4100103,Abatiá - PR,2013,3100,23250,20576,7500,7500,soja,15937620,0.001459
4,4100103,Abatiá - PR,2014,1460,12330,12774,8445,8445,soja,14913173,0.000827


In [5]:
# Perda de Área de plantio por ano (Risco climático) pro ano
df["perda_area_pct"] = (df["area_plantada"] - df["area_colhida"]) / df["area_plantada"].replace(0, np.nan)
df["perda_area_pct"] = df["perda_area_pct"].fillna(0.0)
df["perda_area_pct"] = df["perda_area_pct"].clip(lower=0.0) # Clip: todo numero negativo é transformado em 0.0, evita inconsistencia da base

In [7]:
df.head()

,municipio_codigo,municipio_nome,ano,rendimento_medio,quantidade_produzida,valor_producao,area_colhida,area_plantada,produto,total_prod_ano,market_share,perda_area_pct
0,4100103,Abatiá - PR,2010,2590,17820,8713,6880,6880,soja,14091829,0.001265,0.0
1,4100103,Abatiá - PR,2011,3350,23048,15557,6880,6880,soja,15457911,0.001491,0.0
2,4100103,Abatiá - PR,2012,2320,16472,14001,7100,7100,soja,10937896,0.001506,0.0
3,4100103,Abatiá - PR,2013,3100,23250,20576,7500,7500,soja,15937620,0.001459,0.0
4,4100103,Abatiá - PR,2014,1460,12330,12774,8445,8445,soja,14913173,0.000827,0.0


In [ ]:
def calculate_slope( series: pd.Series) -> float:
    """
    Calcula a inclinação linear da série histórica (Slope).
    Utiliza o índice da série (ano) como variável independente.
    """
    y = series.astype(float).values
    if len(y) < 2 or np.all(np.isnan(y)) or np.sum(y) == 0:
        return 0.0

    x = series.index.astype(float).values

    # Remove eventuais nulos de X e Y
    mask = ~np.isnan(y) & ~np.isnan(x)
    if np.sum(mask) < 2:
        return 0.0

    slope, _ = np.polyfit(x[mask], y[mask], 1)
    return float(slope)

def calculate_cagr(series: pd.Series) -> float:
    """
    Calcula a taxa de crescimento anual composta (CAGR) entre o primeiro e último ano
    com dados maiores que zero, usando o índice da série (ano) para o número de anos decorridos.
    """
    # Filtra valores maiores que zero e remove NaNs
    valid_series = series[series > 0].dropna()
    if len(valid_series) < 2:
        return 0.0

    v_ini = valid_series.iloc[0]
    v_fin = valid_series.iloc[-1]

    y_ini = int(valid_series.index[0])
    y_fin = int(valid_series.index[-1])

    n_years = y_fin - y_ini

    if n_years <= 0 or v_ini <= 0:
        return 0.0

    cagr = (v_fin / v_ini) ** (1.0 / n_years) - 1.0
    return float(cagr)

features_list = []


grouped = df.groupby(["municipio_codigo", "municipio_nome", "produto"])
grouped.mean()
# grouped.get_group((4100103, "Abatiá - PR", "soja"))
for (m_cod, m_nome, prod), group in grouped:
    group = group.sort_values("ano")



    mean_area = group["area_plantada"].mean()
    mean_yield = group["rendimento_medio"].mean()
    mean_value = group["valor_producao"].mean()
    mean_prod = group["quantidade_produzida"].mean()

    # Volatilidade (Coeficiente de Variação) da produção
    std_prod = group["quantidade_produzida"].astype(float).std()
    cv_prod = (std_prod / mean_prod) if mean_prod > 0 else 0.0

    if np.isnan(cv_prod):
        cv_prod = 0.0


    prod_series = group.set_index("ano")["quantidade_produzida"]
    yield_series = group.set_index("ano")["rendimento_medio"]

    cagr_prod = calculate_cagr(prod_series)
    cagr_yield = calculate_cagr(yield_series)
    slope_prod = calculate_slope(prod_series)

    mean_market_share = group["market_share"].mean()
    mean_area_loss = group["perda_area_pct"].mean()


    features_list.append(
        {
            "municipio_codigo": m_cod,
            "municipio_nome": m_nome,
            "produto": prod,
            "prod_media": mean_prod,
            "area_media": mean_area,
            "rendimento_medio_med": mean_yield,
            "valor_producao_medio": mean_value,
            "volatilidade_prod": cv_prod,
            "cagr_producao": cagr_prod,
            "cagr_rendimento": cagr_yield,
            "trend_slope_producao": slope_prod,
            "market_share_medio": mean_market_share,
            "perda_area_media": mean_area_loss,
        })


    # # print(f"prod_series: {prod_series}")
    # print(f"yield_series: {yield_series}")


    # print(f"m_cod: {m_cod}")
    # print(f"m_nome: {m_nome}")
    # print(f"prod: {prod}")
    # # print(group)


df_features = pd.DataFrame(features_list)
df_features


,municipio_codigo,municipio_nome,produto,prod_media,area_media,rendimento_medio_med,valor_producao_medio,volatilidade_prod,cagr_producao,cagr_rendimento,trend_slope_producao,market_share_medio,perda_area_media
0,4100103,Abatiá - PR,milho,21400.600000,4098.333333,5797.533333,14555.533333,0.805873,0.022440,-0.019616,0.022440,1.336364e-03,0.000000
1,4100103,Abatiá - PR,soja,28033.666667,9293.333333,2983.533333,42277.666667,0.292888,0.048450,0.023798,0.048450,1.654275e-03,0.000000
2,4100103,Abatiá - PR,trigo,6993.466667,3049.333333,1846.600000,5305.333333,0.763310,-0.025751,-0.007851,-0.025751,2.300003e-03,0.000000
3,4100202,Adrianópolis - PR,milho,9004.400000,1723.000000,5837.000000,4922.200000,0.397340,-0.164221,0.027434,-0.164221,6.107541e-04,0.000000
4,4100202,Adrianópolis - PR,soja,8.733333,2.533333,459.266667,22.533333,2.646170,-0.128571,-0.031714,-0.128571,4.341073e-07,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1192,4128708,Vitorino - PR,soja,69065.733333,20712.666667,3323.666667,101105.533333,0.253568,0.027235,0.008956,0.027235,4.023279e-03,0.000000
1193,4128708,Vitorino - PR,trigo,12277.333333,4133.333333,3049.866667,10216.266667,0.284036,-0.015041,-0.021724,-0.015041,4.278713e-03,0.005714
1194,4128807,Xambrê - PR,milho,309.200000,149.933333,1861.466667,181.200000,1.391322,0.186031,0.001589,0.186031,1.899522e-05,0.000000
1195,4128807,Xambrê - PR,soja,706.400000,376.800000,1916.666667,1203.600000,0.816487,0.115080,-0.004553,0.115080,3.849453e-05,0.004246
